# Convert TRELLIS.2 Mapper Checkpoints

Convert legacy `WindowGridFeatMapper` checkpoints into the current `Swin3DLatentMapper` checkpoint format. Output checkpoint filenames start with `trellis2_`.

Persistence rule: this notebook saves exactly the keys returned by the current model's `state_dict()`. Non-persistent buffers such as `nbr_offsets` and `pose_conditioner.t_freqs` are intentionally not saved. Persistent buffers such as `pos_pe.freqs`, RoPE `ang_freqs`, and `edge_head.pe.freqs` are saved.

In [1]:
import os
from dataclasses import asdict, replace
from pathlib import Path

import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "symtrellis").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from symtrellis.mapper import Swin3DLatentMapper, swin_3d_latent_mapper_config

In [2]:
CKPT_SPECS = [
    {
        "name": "sparse_structure",
        "old_path_env": "SYMTRELLIS_LEGACY_SS_MAPPER_CKPT",
        "old_path": Path(os.environ["SYMTRELLIS_LEGACY_SS_MAPPER_CKPT"]),
        "new_path": REPO_ROOT / "checkpoints" / "trellis2_sparse_structure_swin3d_latent_mapper_base.pt",
    },
    {
        "name": "shape_latent",
        "old_path_env": "SYMTRELLIS_LEGACY_SHAPE_MAPPER_CKPT",
        "old_path": Path(os.environ["SYMTRELLIS_LEGACY_SHAPE_MAPPER_CKPT"]),
        "new_path": REPO_ROOT / "checkpoints" / "trellis2_shape_latent_swin3d_latent_mapper_small.pt",
    },
]

NAME_REPLACEMENTS = (
    ("pose_cond.", "pose_conditioner."),
    (".a_norm1.", ".norm1_src."),
    (".a_self_attn.", ".self_attn_src."),
    (".a_norm2.", ".norm2_src."),
    (".a_ffn.", ".ffn_src."),
    (".a_norm_cross.", ".norm_cross_src."),
    (".b_norm1.", ".norm1_dst."),
    (".b_cross_attn.", ".cross_attn_dst."),
    (".b_norm2.", ".norm2_dst."),
    (".b_self_attn.", ".self_attn_dst."),
    (".b_norm3.", ".norm3_dst."),
    (".b_ffn.", ".ffn_dst."),
    ("rope.inv_freqs", "rope.ang_freqs"),
    ("lowrank_head.", "coeff_head."),
    ("delta_logit_layer.", "weight_logit_layer."),
)

In [3]:
conversion_results = []

for spec in CKPT_SPECS:
    old_ckpt = torch.load(spec["old_path"], map_location="cpu")
    old_cfg = old_ckpt["config"]

    cfg = swin_3d_latent_mapper_config(
        scale=old_cfg["model_scale"],
        latent_dim=old_cfg["feat_dim"],
        lowrank_rank=old_cfg["lowrank_rank"],
    )
    cfg = replace(cfg, attn_backend=old_cfg["attn_backend"])
    model = Swin3DLatentMapper(cfg)
    target_state = model.state_dict()

    new_state = {}
    dropped_state_keys = []

    for old_name, value in old_ckpt["model"].items():
        if old_name == "possible_displacement":
            dropped_state_keys.append(old_name)
            continue

        new_name = old_name
        for old_prefix, new_prefix in NAME_REPLACEMENTS:
            new_name = new_name.replace(old_prefix, new_prefix)

        if new_name not in target_state:
            raise KeyError(f"{spec['name']}: unmapped key {old_name!r} -> {new_name!r}")
        if target_state[new_name].shape != value.shape:
            raise RuntimeError(
                f"{spec['name']}: shape mismatch {old_name!r} -> {new_name!r}: "
                f"{tuple(value.shape)} != {tuple(target_state[new_name].shape)}"
            )

        new_state[new_name] = value

    missing_keys = sorted(set(target_state) - set(new_state))
    if missing_keys:
        raise RuntimeError(f"{spec['name']}: missing keys: {missing_keys}")

    model.load_state_dict(new_state, strict=True)
    spec["new_path"].parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "format": "symtrellis.swin3d_latent_mapper",
            "name": f"trellis2_{spec['name']}",
            "config": asdict(cfg),
            "model": new_state,
            "source": {
                "checkpoint_env": spec["old_path_env"],
                "epoch": old_ckpt.get("epoch"),
                "global_step": old_ckpt.get("global_step"),
                "model_scale": old_cfg["model_scale"],
                "grid_size": old_cfg["grid_size"],
                "feat_dim": old_cfg["feat_dim"],
                "lowrank_rank": old_cfg["lowrank_rank"],
                "dropped_state_keys": dropped_state_keys,
                "non_persistent_buffers_not_saved": ["nbr_offsets", "pose_conditioner.t_freqs"],
            },
        },
        spec["new_path"],
    )

    conversion_results.append(
        {
            "name": spec["name"],
            "scale": old_cfg["model_scale"],
            "old_path": f"<{spec['old_path_env']}>",
            "new_path": str(spec["new_path"]),
            "num_state_keys": len(new_state),
            "dropped_state_keys": dropped_state_keys,
        }
    )

conversion_results

[{'name': 'sparse_structure',
  'scale': 'base',
  'old_path': '<SYMTRELLIS_LEGACY_SS_MAPPER_CKPT>',
  'new_path': 'checkpoints/trellis2_sparse_structure_swin3d_latent_mapper_base.pt',
  'num_state_keys': 396,
  'dropped_state_keys': ['possible_displacement']},
 {'name': 'shape_latent',
  'scale': 'small',
  'old_path': '<SYMTRELLIS_LEGACY_SHAPE_MAPPER_CKPT>',
  'new_path': 'checkpoints/trellis2_shape_latent_swin3d_latent_mapper_small.pt',
  'num_state_keys': 302,
  'dropped_state_keys': ['possible_displacement']}]